In [9]:
import re
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Specify the path to your text file in Google Drive
# Change this path to match your exact folder structure and filename
file_path = '/content/drive/MyDrive/physics_212_syllabus.txt'

try:
    with open(file_path, 'r', encoding='utf-8') as file:
        syllabus_text = file.read()
    print(f"Successfully loaded file from: {file_path}")
except FileNotFoundError:
    print(f"Error: Could not find the file at '{file_path}'. Please verify your file path.")
    syllabus_text = ""

# 3. Split into clean paragraph list
def paragraph_split(syllabus_text):
  syllabus_paragraphs = [p.strip() for p in syllabus_text.strip().split('\n\n') if p.strip()]
  print(f"Dataset successfully parsed: {len(syllabus_paragraphs)} paragraphs found.\n")
  return syllabus_paragraphs

# ==========================================
# STRATEGY 1: Fixed Length Chunking (Sentence-Boundary Aware)
# ==========================================
def fixed_length_chunking(syllabus_text, max_chars_per_chunk=450):
    paragraphs = paragraph_split(syllabus_text)
    chunks = []
    current_chunk = ""

    def split_sentences(text):
        return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

    for para in paragraphs:
        sentences = split_sentences(para)
        for sentence in sentences:
            if len(current_chunk) + len(sentence) + 1 > max_chars_per_chunk and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = sentence
            else:
                current_chunk += (" " + sentence) if current_chunk else sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

fixed_chunks = fixed_length_chunking(syllabus_text, max_chars_per_chunk=450)
print(f"--- Strategy 1: Fixed Length Chunking ({len(fixed_chunks)} chunks created) ---")
if fixed_chunks:
    print(f"Sample Chunk 1:\n{fixed_chunks[0]}\n")

# ==========================================
# STRATEGY 2: Overlapping Paragraph Chunking
# ==========================================
def overlapping_paragraph_chunking(syllabus_text):
    paragraphs = paragraph_split(syllabus_text)
    chunks = []
    for i in range(len(paragraphs) - 1):
        combined = f"{paragraphs[i]}\n\n{paragraphs[i+1]}"
        chunks.append(combined)
    return chunks

overlapping_chunks = overlapping_paragraph_chunking(syllabus_text)
print(f"--- Strategy 2: Overlapping Paragraph Chunking ({len(overlapping_chunks)} chunks created) ---")
if overlapping_chunks:
    print(f"Sample Chunk 1:\n{overlapping_chunks[0]}\n")

# ==========================================
# STRATEGY 3: Hybrid/Specialized Chunking (Currently working on it)
# ==========================================

def header_chunking(syllabus_text, header_pattern=r'^###\s+'):
    paragraphs = paragraph_split(syllabus_text)
    chunks = []
    current_chunk = []

    for para in paragraphs:
        # Check if the current paragraph is a header
        if re.match(header_pattern, para):
            # If we already have an active chunk, save it before starting a new one
            if current_chunk:
                chunks.append("\n\n".join(current_chunk))
                current_chunk = [] # Reset for the new chunk

        # Add the paragraph (whether header or body) to the current active chunk
        current_chunk.append(para)

    # Don't forget to append the final accumulated chunk after the loop ends
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    print(f"Successfully created {len(chunks)} header-based chunks.\n")
    return chunks


header_chunks = header_chunking(syllabus_text)

# Example: Print the first chunk to verify
print(f"--- Strategy 3: Overlapping Paragraph Chunking ({len(header_chunks)} chunks created) ---")
if overlapping_chunks:
    print(f"Sample Chunk 1:\n{header_chunks[0]}\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully loaded file from: /content/drive/MyDrive/physics_212_syllabus.txt
Dataset successfully parsed: 30 paragraphs found.

--- Strategy 1: Fixed Length Chunking (55 chunks created) ---
Sample Chunk 1:
﻿### 1. Course Overview & Objectives
Welcome to Physics 212, an intensive and comprehensive exploration into the fundamental principles of electricity and magnetism. Throughout this course, students will investigate the physical laws governing electric charges, electric fields, magnetic fields, electromagnetic induction, and Maxwell's equations.

Dataset successfully parsed: 30 paragraphs found.

--- Strategy 2: Overlapping Paragraph Chunking (29 chunks created) ---
Sample Chunk 1:
﻿### 1. Course Overview & Objectives
Welcome to Physics 212, an intensive and comprehensive exploration into the fundamental principles of electricity and magnetism. Throughout

#### 1. How Paragraphs Are Defined
We decided to define a paragraph to be something separated by a blank line. We also added code to make sure we do not count multiple empty lines as a paragraph by filtering out such occasions.

#### 2. How Overlap is Implemented

Adjacent Pairing: The for loop iterates through the list of paragraphs stopping one index short (len(paragraphs) - 1) so it can always look ahead to the next paragraph.

Mechanism:

Chunk 1 combines Paragraph 1 and Paragraph 2 (P1 + P2)

Chunk 2 combines Paragraph 2 and Paragraph 3 (P2 + P3)

Chunk 3 combines Paragraph 3 and Paragraph 4 (P3 + P4)

Etc etc


Ensures that if a search query or concept spans across a paragraph break, the semantic context isn't lost or cut off mid-thought.

In [10]:
# Install required libraries if not already present in your Colab environment
# !pip install sentence-transformers faiss-cpu
!pip install faiss-cpu sentence-transformers transformers accelerate
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

def build_multi_model_vector_indices(chunks: list[str]) -> dict:
    if not chunks:
        raise ValueError("The input list of chunks is empty.")

    # 1. Define Hugging Face models for Small, Medium, and Large tiers
    model_configs = {
        "small": "sentence-transformers/all-MiniLM-L6-v2",    # Fast & Lightweight (~90MB)
        "medium": "sentence-transformers/all-mpnet-base-v2",  # High quality, balanced (~420MB)
        "large": "BAAI/bge-large-en-v1.5"                     # SOTA high-capacity model (~1.3GB)
    }

    searchable_structures = {}

    for size, model_name in model_configs.items():
        print(f"[{size.upper()}] Loading model: {model_name}...")
        model = SentenceTransformer(model_name)

        # Step 1: Convert each chunk into embeddings
        print(f"[{size.upper()}] Encoding {len(chunks)} chunks into embeddings...")
        embeddings = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

        # Normalize vectors for cosine similarity
        faiss.normalize_L2(embeddings)
        embedding_dim = embeddings.shape[1]

        # Step 2: Store embeddings in a searchable FAISS structure (Inner Product / Cosine Similarity)
        index = faiss.IndexFlatIP(embedding_dim)
        index.add(embeddings.astype(np.float32))

        searchable_structures[size] = {
            "model_name": model_name,
            "model": model,
            "index": index,
            "chunks": chunks,
            "dimension": embedding_dim
        }
        print(f"[{size.upper()}] Successfully indexed vectors with dimension {embedding_dim}.\n")

    return searchable_structures

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 12.1 MB/s  0:00:01


In [ ]:
import numpy as np

def retrieve_relevant_chunks(vector_stores: dict, model_tier: str, query: str, top_k: int = 3) -> list[dict]:
    """
    Performs similarity search and top-k retrieval for a given query.

    Parameters:
        vector_stores (dict): The dictionary returned by build_multi_model_vector_indices.
        model_tier (str): Which model to use ('small', 'medium', or 'large').
        query (str): The search query text.
        top_k (int): Number of top relevant chunks to retrieve.

    Returns:
        list[dict]: A ranked list of dictionaries containing similarity scores and chunk text.
    """
    # Validate model tier
    if model_tier not in vector_stores:
        raise ValueError(f"Invalid tier '{model_tier}'. Choose from: {list(vector_stores.keys())}")

    store = vector_stores[model_tier]
    model = store["model"]
    index = store["index"]
    chunks = store["chunks"]

    # 1. Convert Query to Embedding
    # Encode the query string into a vector array
    query_embedding = model.encode([query], convert_to_numpy=True)

    # 2. Similarity Search Prep (L2 Normalization)
    # Since we used IndexFlatIP (Inner Product) with normalized vectors during indexing,
    # we must normalize the query vector to compute true cosine similarity scores.
    faiss.normalize_L2(query_embedding)

    # 3. Perform Similarity Search
    # index.search returns distance/score matrices and corresponding array indices
    distances, indices = index.search(query_embedding.astype(np.float32), top_k)

    # 4. Top-K Retrieval of Relevant Chunks
    retrieved_results = []
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
        if idx == -1:  # FAISS returns -1 if index contains fewer items than top_k
            continue

        retrieved_results.append({
            "rank": rank + 1,
            "similarity_score": float(score), # Higher score means higher similarity (range ~[-1, 1])
            "chunk_index": int(idx),
            "content": chunks[idx]
        })

    return retrieved_results

# --- Example Usage ---
# Assuming 'vector_stores' was created from your syllabus chunks:
#query_text = "What is the penalty for submitting homework late?"
#top_results = retrieve_relevant_chunks(vector_stores, model_tier="medium", query=query_text, top_k=2)

#for res in top_results:
#    print(f"Rank {res['rank']} (Score: {res['similarity_score']:.4f})")
#    print(f"Content:\n{res['content']}\n{'-'*40}")

Using 52 chunks generated from fixed_length_chunking for embedding.

Loading [Small] model -> all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Small] Index built successfully with 52 fixed-length chunks!

Loading [Medium] model -> all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[Medium] Index built successfully with 52 fixed-length chunks!

Loading [Large] model -> BAAI/bge-large-en-v1.5...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Large] Index built successfully with 52 fixed-length chunks!

Loading LLM [Qwen/Qwen3-4B-Instruct-2507]...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LLM loaded successfully!

Running retrieval using [Medium] tier model over fixed-length chunks...
Generating response from Qwen...

USER QUERY: When are the weekly homework assignments due, and when does the midterm exam take place?
The weekly homework assignments are due on Thursday at 12:00 PM, right before class begins. The midterm exam takes place during week eight.


In [12]:
!pip install -U transformers accelerate torch
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def generate_rag_response(query: str, retrieved_chunks: list[dict]) -> str:
    """
    Constructs a RAG prompt using system instructions, retrieved context, and user query,
    then uses Qwen3-4B-Instruct-2507 to generate the final output.

    Parameters:
        query (str): The user's search or question string.
        retrieved_chunks (list[dict]): Top-k chunks returned by the retrieval pipeline.

    Returns:
        str: The generated model response.
    """
    model_id = "Qwen/Qwen3-4B-Instruct-2507"

    print(f"Loading tokenizer and model: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Load model with automatic device mapping and bfloat16 for efficiency
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )

    # 1. Format the Retrieved Context
    context_text = "\n\n".join([
        f"--- Syllabus Section {i+1} ---\n{chunk['content']}"
        for i, chunk in enumerate(retrieved_chunks)
    ])

    # 2. Define System Instructions
    system_instruction = (
        "You are an expert AI teaching assistant for Physics 212. "
        "Answer the user's question accurately and concisely using ONLY the provided syllabus context below. "
        "If the answer cannot be found directly within the context, politely state that the information is not available in the course syllabus."
    )

    # 3. Construct the User Prompt Payload
    user_content = f"""Here is the retrieved course syllabus context:
{context_text}

Based strictly on the above context, please answer the following question:
{query}"""

    # 4. Format into Chat Template Messages
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]

    # 5. Tokenize using Qwen's official chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # 6. Generate Response (using recommended sampling parameters for Qwen3-Instruct)
    print("Generating response with Qwen3-4B-Instruct-2507...")
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True
    )

    # Decode only the newly generated tokens (slice off the input prompt tokens)
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response

# --- Example End-to-End Usage ---
# 1. Assume you ran your retrieval step:
# top_results = retrieve_relevant_chunks(vector_stores, model_tier="medium", query="What is the late homework penalty?", top_k=2)

# 2. Generate the final answer:
# final_answer = generate_rag_response("What is the penalty for late homework?", top_results)
# print("\n--- Final Generated Answer ---\n")
# print(final_answer)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 27.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 2.8 MB/s  0:01:35
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 11.6 MB/s  0:00:37
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.1 MB/s  0:00:20
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 5.4 MB/s  0:00:54
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 2.8 MB/s  0:00:45
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 3.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 5.0 MB/s  0:01:45
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 5.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 5.7 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 5.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 3.8 MB/s  0:00:41
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 3.9 MB/s  0:00:00
   ━━━━━╺━━━━━━